In [388]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "input.txt")

with open("dataset.txt", 'r', encoding="utf8") as f:
    text = f.read()

with open("input.txt", "r", encoding="utf8") as f:
    input_text = f.read()

In [389]:
f"Length of text dataset: {len(text)}"

'Length of text dataset: 26987'

In [390]:
f"Length of input_text dataset: {len(input_text)}"

'Length of input_text dataset: 1115394'

In [ ]:
print(text[:1000])

In [392]:
chars = list(set(text))
vocab_size = len(chars)
print("vocab size:", vocab_size)
print("".join(chars))


vocab size: 84
🙁Dhf*oEX0mVp-OJu”6🙂t😉A?xU8
″F(PIkjbG:'Sgsz—"ywK.ieaB>)<!H/9q;l5Tn&’r,“N dCY…WvM12LcR


In [393]:
char2id = { ch : idx for idx, ch in enumerate(chars) }
id2char = { idx : ch for idx, ch in enumerate(chars) }

print(char2id)
print("\n\n\n",id2char)

{'🙁': 0, 'D': 1, 'h': 2, 'f': 3, '*': 4, 'o': 5, 'E': 6, 'X': 7, '0': 8, 'm': 9, 'V': 10, 'p': 11, '-': 12, 'O': 13, 'J': 14, 'u': 15, '”': 16, '6': 17, '🙂': 18, 't': 19, '😉': 20, 'A': 21, '?': 22, 'x': 23, 'U': 24, '8': 25, '\n': 26, '″': 27, 'F': 28, '(': 29, 'P': 30, 'I': 31, 'k': 32, 'j': 33, 'b': 34, 'G': 35, ':': 36, "'": 37, 'S': 38, 'g': 39, 's': 40, 'z': 41, '—': 42, '"': 43, 'y': 44, 'w': 45, 'K': 46, '.': 47, 'i': 48, 'e': 49, 'a': 50, 'B': 51, '>': 52, ')': 53, '<': 54, '!': 55, 'H': 56, '/': 57, '9': 58, 'q': 59, ';': 60, 'l': 61, '5': 62, 'T': 63, 'n': 64, '&': 65, '’': 66, 'r': 67, ',': 68, '“': 69, 'N': 70, ' ': 71, 'd': 72, 'C': 73, 'Y': 74, '…': 75, 'W': 76, 'v': 77, 'M': 78, '1': 79, '2': 80, 'L': 81, 'c': 82, 'R': 83}



 {0: '🙁', 1: 'D', 2: 'h', 3: 'f', 4: '*', 5: 'o', 6: 'E', 7: 'X', 8: '0', 9: 'm', 10: 'V', 11: 'p', 12: '-', 13: 'O', 14: 'J', 15: 'u', 16: '”', 17: '6', 18: '🙂', 19: 't', 20: '😉', 21: 'A', 22: '?', 23: 'x', 24: 'U', 25: '8', 26: '\n', 27: '″', 28: 

In [394]:
def encode(text):
    return [ char2id[char] if char in char2id else "[UNK]" for char in text]

def decode(ids):
    return "".join([ id2char[id] if id in id2char else "[UNK]" for id in ids ])

encoded = encode("text")
print(encoded)

decoded = decode(encoded)
print(decoded)

[19, 49, 23, 19]
text


In [395]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.set_default_device(device)


data = torch.tensor(encode(text), dtype = torch.long)

In [396]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [397]:
block_size = 8

x = train_data[:block_size]
print("x: ", x, len(x))
y = train_data[1: block_size + 1]
print("\ny: ", y, len(y))

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(context.data, "target: ", target.data)


x:  tensor([56, 49, 36, 71, 31, 71, 45, 50], device='mps:0') 8

y:  tensor([49, 36, 71, 31, 71, 45, 50, 64], device='mps:0') 8
tensor([56], device='mps:0') target:  tensor(49, device='mps:0')
tensor([56, 49], device='mps:0') target:  tensor(36, device='mps:0')
tensor([56, 49, 36], device='mps:0') target:  tensor(71, device='mps:0')
tensor([56, 49, 36, 71], device='mps:0') target:  tensor(31, device='mps:0')
tensor([56, 49, 36, 71, 31], device='mps:0') target:  tensor(71, device='mps:0')
tensor([56, 49, 36, 71, 31, 71], device='mps:0') target:  tensor(45, device='mps:0')
tensor([56, 49, 36, 71, 31, 71, 45], device='mps:0') target:  tensor(50, device='mps:0')
tensor([56, 49, 36, 71, 31, 71, 45, 50], device='mps:0') target:  tensor(64, device='mps:0')


In [398]:
torch.manual_seed(42)
batch_size = 4

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : block_size + i] for i in ix])
    y = torch.stack([data[i + 1 : block_size + 1 + i] for i in ix])
    return x,y

xb, yb = get_batch("train")
print("shape: ", xb.shape)
print("inputs: \n", xb);
print('\n\n')
print("shape: ",yb.shape)
print("targets: \n", yb)



shape:  torch.Size([4, 8])
inputs: 
 tensor([[45, 48, 40,  2, 71, 31, 71, 45],
        [71,  2, 49, 67, 49, 71, 31, 71],
        [38,  2, 49, 36, 71, 13, 32, 50],
        [45, 47, 26, 38,  2, 49, 36, 71]], device='mps:0')



shape:  torch.Size([4, 8])
targets: 
 tensor([[48, 40,  2, 71, 31, 71, 45, 50],
        [ 2, 49, 67, 49, 71, 31, 71, 32],
        [ 2, 49, 36, 71, 13, 32, 50, 44],
        [47, 26, 38,  2, 49, 36, 71, 21]], device='mps:0')


In [399]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print("input:", context.tolist(),"  target: ", target.tolist())

input: [45]   target:  48
input: [45, 48]   target:  40
input: [45, 48, 40]   target:  2
input: [45, 48, 40, 2]   target:  71
input: [45, 48, 40, 2, 71]   target:  31
input: [45, 48, 40, 2, 71, 31]   target:  71
input: [45, 48, 40, 2, 71, 31, 71]   target:  45
input: [45, 48, 40, 2, 71, 31, 71, 45]   target:  50
input: [71]   target:  2
input: [71, 2]   target:  49
input: [71, 2, 49]   target:  67
input: [71, 2, 49, 67]   target:  49
input: [71, 2, 49, 67, 49]   target:  71
input: [71, 2, 49, 67, 49, 71]   target:  31
input: [71, 2, 49, 67, 49, 71, 31]   target:  71
input: [71, 2, 49, 67, 49, 71, 31, 71]   target:  32
input: [38]   target:  2
input: [38, 2]   target:  49
input: [38, 2, 49]   target:  36
input: [38, 2, 49, 36]   target:  71
input: [38, 2, 49, 36, 71]   target:  13
input: [38, 2, 49, 36, 71, 13]   target:  32
input: [38, 2, 49, 36, 71, 13, 32]   target:  50
input: [38, 2, 49, 36, 71, 13, 32, 50]   target:  44
input: [45]   target:  47
input: [45, 47]   target:  26
input:

In [ ]:
nn = torch.nn
F = nn.functional

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:  
            B, T, C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #idx is B,T array in the cur context
        for _ in range(max_new_tokens):

            #get predictions
            logits, loss = self(idx)

            #focus on the last time step
            logits = logits[:, -1, :] # --> (B, C)

            #apply softmax
            probs = F.softmax(logits, dim = -1) # --> (B, C)

            #sample from dist
            idx_next = torch.multinomial(probs, num_samples=1) # --> (B, 1)

            #append sampled index to the running sequence
            idx = torch.cat((idx,idx_next), dim = 1) # --> (B, T + 1)
        return idx
            


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape, "\n\n" ,loss)

idx = torch.zeros((1,1), dtype=torch.long)

decode(m.generate(idx, max_new_tokens=100)[0].tolist())


torch.Size([32, 84]) 

 tensor(4.9167, device='mps:0', grad_fn=<NllLossBackward0>)


'🙁uTS“&meMWrA🙁urSc:Wq😉’qSo&c&zwBCBu″fF1Y*GdJ“S&MIY/01(bKWSJ🙂dRH…>JEE5,oppwVIScunRY’>T&Aae!HRT🙂V&<…2j)😉'

In [401]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [416]:
batch_size = 32

for steps in range(1000):
    xb, yb = get_batch("train")

    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item())




2.512101888656616
2.5008809566497803
2.545185089111328
2.4398040771484375
2.673552989959717
2.466421604156494
2.5356667041778564
2.431602954864502
2.5173583030700684
2.4599947929382324
2.4541890621185303
2.508288860321045
2.4913673400878906
2.5593156814575195
2.5521979331970215
2.417755126953125
2.4288430213928223
2.445310354232788
2.5042030811309814
2.4617996215820312
2.500129222869873
2.5285816192626953
2.6385998725891113
2.446636199951172
2.5494890213012695
2.5651440620422363
2.4812707901000977
2.4200124740600586
2.4896626472473145
2.474977493286133
2.6066620349884033
2.3912665843963623
2.573725700378418
2.442369222640991
2.5413591861724854
2.621767997741699
2.543269634246826
2.471144199371338
2.507450580596924
2.6182761192321777
2.614436388015747
2.584132194519043
2.4002761840820312
2.550133228302002
2.684814453125
2.578372001647949
2.4461798667907715
2.4460325241088867
2.4289655685424805
2.5420119762420654
2.427031993865967
2.5405545234680176
2.539926767349243
2.499788284301758
2.

In [420]:
idx = torch.zeros((1,1), dtype=torch.long)

decode(m.generate(idx, max_new_tokens=100)[0].tolist())

'🙁’sE″cTLwwhmohe pp) d ;) a at\nShtich t monginbe:Hesh ss oung tSheMit?\nH6k!loungk? yoir y ank fhit soo'